# Day 11 — Iterators, Generators, Decorators & Context Managers (4 Nov)

*Rebuilt from the master Google Doc session (lesson text and every code example). Code cells are not executed: several need Postgres, a running server or files from the original starter zip.*

## Today at a glance

- Goal: understand how a for loop really works (iterators), process data of any size one record at a time (generators), add behaviour such as retries to any function without editing it (decorators), and guarantee cleanup even when something fails (context managers).

- Deliverable: day11/retry.py (a retry decorator with exponential backoff), day11/stream_export.py (a streaming reader that turns a large JSONL CMS export into a CSV), day11/test_day11.py with 12 or more green tests, and day11/dsa.py. The sample export and a big-file generator are given.

- Where this sits: Week 2, Phase 1 (Foundation), day 4 of 7. These four ideas are everywhere in production Python: FastAPI uses decorators for every route (Week 4), the Claude SDK streams responses with iterators (Week 5), RAG ingestion streams documents through generators (Week 6), and database sessions are context managers.

Time split (5h):

| Block | Time | What you do |
|---|---|---|
| Theory | 1.5h | Concepts 1 to 4, typing every example in a scratch folder |
| Hands-on | 2.5h | Part A: retry.py; Part B: stream_export.py; tests for both |
| DSA | 0.5h | Moving average of a stream, flatten a nested list |
| Revision | 0.5h | Quiz, Remember cells in the notebook, the memory check |

Open items first: the repo on GitHub still shows day01/ to day04/, and day04/ still holds the starter TODOs. Days 5 to 10 are not pushed. If that work is on your laptop, push it before you start (one branch per day is fine). The Day 7 rule still holds: the weekly score on Day 14 reads the repo. Today's task is self-contained, so you can do it even if earlier days are unfinished.

Where to type today's examples: a scratch folder such as ~/scratch/day11, with your venv active. Everything today is in the standard library except requests (Day 6), used in one example.

## Recap (2 minutes, from memory)

- A test is named check_export in a file called test_export.py. What does pytest do with it, and what single change fixes it?

- What is the difference between a fixture that uses return and one that uses yield? When do you need yield?

- A module does from time import sleep and calls sleep(2). You want a test that does not really wait. Which name do you patch with monkeypatch, and why does patching time.sleep not work?

Answers are on the last page of today's session.

## Concept 1: Iterators and the iteration protocol

### Why it matters for an FDE

Every time you loop over a list, a file, a database cursor or a paginated API, Python uses the same small protocol underneath. Once you understand it, three things that confuse beginners stop being mysterious: why a second loop over some objects gives nothing, why zip() prints as <zip object> instead of a list, and how a file with ten million lines can be read without loading it.

It also matters for reading other people's code. Customer codebases and SDKs (the Claude SDK, boto3 for AWS, database drivers) return iterators all the time, for streamed responses and paginated results. If you call list() on them without thinking, you either lose data or load far too much.

### How it works

Two words carry the whole idea. An iterable is anything you can loop over: a list, a string, a dict, a file. An iterator is the object that does the walking: it remembers a position and hands out the next item when asked. The built-in iter(x) asks an iterable for a fresh iterator, and next(it) asks the iterator for the next item. When nothing is left, the iterator raises StopIteration.

A for loop is just that, done for you: it calls iter() once, then next() again and again, and stops quietly at StopIteration. JavaScript has the same design: [Symbol.iterator]() returns an iterator whose next() gives {value, done}. Python signals the end with an exception instead of a done flag.

In [ ]:
# The iteration protocol by hand: what a for loop does under the hood
locales = ["en-us", "fr-fr", "hi-in"]  # a list is an ITERABLE
it = iter(locales)  # ask it for an ITERATOR: an object with a position
print(next(it))  # -> en-us
print(next(it))  # -> fr-fr
print(next(it))  # -> hi-in
try:
    next(it)  # nothing left: the iterator signals "done" by raising
except StopIteration:
    print("done")  # -> done
# JS: it.next() gives {value, done}; Python raises StopIteration

The important consequence: an iterator is one-shot. It only moves forward and never rewinds. A list is not an iterator; it is an iterable that hands out a new iterator every time you loop, which is why you can loop over a list twice. Many built-ins (zip, map, filter, enumerate, reversed) return iterators, so the second use comes back empty.

In [ ]:
# Iterators are ONE-SHOT; iterables give a fresh iterator each time
uids = ["blt001", "blt002", "blt003"]
it = iter(uids)
print(list(it))  # -> ['blt001', 'blt002', 'blt003']
print(list(it))  # -> []  (already used up; it never rewinds)
print(list(uids), list(uids))  # a list hands out a NEW iterator per loop
# -> ['blt001', 'blt002', 'blt003'] ['blt001', 'blt002', 'blt003']

# Many built-ins return iterators, not lists:
pairs = zip(["a", "b"], [1, 2])  # like a lazy Array.prototype.map pairing
print(pairs)  # -> <zip object at 0x...>  (a zip object, not a list)
print(dict(pairs))  # -> {'a': 1, 'b': 2}
print(dict(pairs))  # -> {}  GOTCHA: the zip was used up by the line above

A file object is its own iterator over lines. Looping over it reads one line at a time from disk, so memory use does not depend on the file size. This is the foundation for today's streaming reader.

In [ ]:
# A file object is an iterator over lines: it reads one line at a time
from pathlib import Path

path = Path("entries.jsonl")
path.write_text('{"uid": "blt001"}\n{"uid": "blt002"}\n', encoding="utf-8")

with open(path, encoding="utf-8") as f:
    print(iter(f) is f)  # -> True  (a file IS its own iterator)
    for line_no, line in enumerate(f, start=1):  # only 1 line in memory
        print(line_no, line.strip())
# -> 1 {"uid": "blt001"}
# -> 2 {"uid": "blt002"}

You can write an iterator yourself with a class that has __iter__ (return the iterator, usually self) and __next__ (return the next item or raise StopIteration). You will rarely do this by hand, because generators in Concept 2 do it in a few lines, but seeing it once shows that there is no magic.

In [ ]:
# Your own iterator class: __iter__ returns self, __next__ gives the next
class PageNumbers:
    """Yields page numbers 1..last, like a CMS paginator."""

    def __init__(self, last: int):
        self.last = last
        self.current = 0  # the position lives INSIDE the iterator

    def __iter__(self):  # like [Symbol.iterator]() in JavaScript
        return self

    def __next__(self) -> int:
        if self.current >= self.last:
            raise StopIteration  # tells the for loop to stop
        self.current += 1
        return self.current


for page in PageNumbers(3):
    print("fetch page", page)
# -> fetch page 1
# -> fetch page 2
# -> fetch page 3

### Real-world FDE example

A customer's Drupal site exposes 1.2 million nodes through a paginated JSON:API. Your migration script wraps the pagination in an iterator: each next() returns one node and fetches the next page only when the current one runs out. The rest of the script just writes for node in all_nodes(session): and never knows about pages, offsets or the links.next URL. When the customer asks you to resume from node 600,000 after a crash, you change one function, not the whole script.

### Common mistakes

- Looping twice over the same iterator (a zip, a map, an open file) and getting nothing the second time. Convert it to a list once if you really need two passes, or recreate it.

- Calling list() on an iterator just to look at it while debugging, and then wondering why the real loop has no data. list() used it up.

- Thinking a StopIteration is an error to log. It is the normal end signal; the for loop handles it.

- Changing a list while looping over it (removing items). The iterator's position gets out of step with the list and items are skipped. Build a new list with a comprehension instead.

## Concept 2: Generators, lazy pipelines that stream data

### Why it matters for an FDE

Customer data is big and you do not control its size. A content export can be 5 GB, a log file 20 GB, a SharePoint crawl millions of documents. Day 4's json.load() reads the whole file into memory at once. That works for a demo and crashes in production. Generators let you process data one item at a time, so memory stays flat whatever the input size, and the first result comes out immediately instead of after the whole file is read.

They also give you a clean way to structure data work: small stages (read, parse, filter, transform, batch), each a few lines, chained into a pipeline. This is how RAG ingestion pipelines are built in Week 6, and how the Claude API's streaming responses arrive in Week 5.

### How it works

A generator function is a normal def that contains yield. Calling it does not run the body; it returns a generator object, which is an iterator. Each next() runs the body until the next yield, hands out that value, and pauses right there with all local variables kept. The next next() resumes on the line after the yield. When the function ends, the generator raises StopIteration for you. JavaScript's function* with yield works the same way.

In [ ]:
# A generator function: "yield" makes Python build the iterator for you
def page_numbers(last: int):
    print("  (generator starts)")  # runs only when the first value is asked
    for n in range(1, last + 1):
        yield n  # hand out n, then PAUSE here until next() is called again
    print("  (generator ends)")  # falling off the end = StopIteration


gen = page_numbers(2)  # nothing runs yet: you only get a generator object
print(type(gen).__name__)  # -> generator
print(next(gen))  # -> "(generator starts)" is printed, then 1
print(next(gen))  # -> 2
print(list(gen))  # -> "(generator ends)" is printed, then []
# JS: the same idea as function* pageNumbers() { yield n } in JavaScript

A generator expression is a comprehension in round brackets instead of square ones. It describes the values but produces them only when asked. The memory difference is not small.

In [ ]:
# Memory: a list holds every item; a generator holds only its position
import sys

squares_list = [n * n for n in range(1_000_000)]  # builds 1M ints now
squares_gen = (n * n for n in range(1_000_000))  # () = generator expression

print(sys.getsizeof(squares_list) > 8_000_000)  # -> True  (about 8 MB)
print(sys.getsizeof(squares_gen))  # -> 200  (bytes, on Python 3.13)
print(sum(squares_gen))  # -> 333332833333500000  (values made one by one)
# The generator's size is the same for 10 items or 10 billion.
# REMEMBER: [ ] builds everything now; ( ) makes items on demand

Generators combine into pipelines. Each stage takes an iterable and yields results, so you can chain them like Unix pipes (Day 9). Nothing happens when you build the chain. Work starts only when something at the end pulls values, and then each record flows through all the stages before the next record is read. The printed order below proves it.

In [ ]:
# A streaming pipeline: each stage pulls ONE record through at a time
import json
from collections.abc import Iterator
from pathlib import Path

path = Path("export.jsonl")
lines = [
    '{"uid": "blt001", "title": "SSO guide", "status": "published"}',
    '{"uid": "blt002", "title": "Draft FAQ", "status": "draft"}',
    "",
    '{"uid": "blt003", "title": "Pricing", "status": "published"}',
]
path.write_text("\n".join(lines) + "\n", encoding="utf-8")


def read_records(p: Path) -> Iterator[dict]:
    # Iterator[dict] = "yields dicts"; the hint for any generator
    with open(p, encoding="utf-8") as f:
        for line in f:
            if line.strip():  # skip blank lines
                yield json.loads(line)  # like JSON.parse(line) in JS


def published(records: Iterator[dict]) -> Iterator[dict]:
    for r in records:
        if r["status"] == "published":
            yield r


def titles(records: Iterator[dict]) -> Iterator[str]:
    for r in records:
        print("  title for", r["uid"])  # shows WHEN each record flows
        yield r["title"]


pipeline = titles(published(read_records(path)))  # nothing read yet
for t in pipeline:  # the for loop pulls records through all 3 stages
    print(t)
# ->   title for blt001
# -> SSO guide
# ->   title for blt003
# -> Pricing

Notice where the file is opened: inside the generator, in a with block. The file stays open while the pipeline runs and closes as soon as the last line has been read. The hint for a generator is Iterator[dict] ("yields dicts"), from collections.abc.

The standard library has tools that work on any iterator without loading it. itertools.islice(it, n) takes the first n items, like .slice() for a stream. itertools.batched(it, n) (Python 3.12 and newer) groups items into tuples of n, which is exactly what bulk APIs want. yield from other hands out every item of another iterable, which is handy for flattening pages into entries.

In [ ]:
# Tools for generators: islice, batched, yield from
from itertools import batched, islice  # batched needs Python 3.12+

uids = (f"blt{n:03}" for n in range(1, 8))  # 7 lazy uids: blt001..blt007
print(list(islice(uids, 2)))  # -> ['blt001', 'blt002']  (first 2 only)
for batch in batched(uids, 2):  # continues where islice stopped
    print(batch)  # tuples of up to 2, like chunks for a bulk API call
# -> ('blt003', 'blt004')
# -> ('blt005', 'blt006')
# -> ('blt007',)  (the last batch can be shorter)


def all_entries(pages):
    for page in pages:
        yield from page  # same as: for e in page: yield e


print(list(all_entries([["e1", "e2"], ["e3"]])))  # -> ['e1', 'e2', 'e3']

### Real-world FDE example

A customer gives you a Contentstack export of 200,000 entries as JSON Lines (one JSON object per line, the format most bulk exports and LLM batch APIs use). Loading it all into a list of dicts took 194 MB of memory in a test for this session; streaming the same file through a generator pipeline into a CSV peaked at 577 KB. On a customer's small VM, or inside a container with a 512 MB limit, that is the difference between a job that runs nightly and one that is killed halfway. In today's task you reproduce this measurement yourself.

### Common mistakes

- Calling list() on a generator "to be safe". It throws away the whole benefit and loads everything. Keep the pipeline lazy until the final consumer.

- Expecting an error at call time. Code inside a generator, including a FileNotFoundError, runs only at the first next(). Today's task tests exactly this.

- Using a generator twice. Like every iterator it is one-shot; the second loop gets nothing, with no error to warn you.

- Using return value to hand out results. In a generator, return ends the iteration; the value is not part of the loop's output. Use yield.

- Returning a generator expression from inside a with open(...) block in a normal function. The function returns, the file closes, and the generator fails later with "I/O operation on closed file". Put the with inside a generator function instead (Check 5 shows this).

## Concept 3: Decorators, adding behaviour without editing the function

### Why it matters for an FDE

Integration code needs the same extra behaviour in many places: retry on 429 and 503, log how long each call took, check that the API key is set, cache results, check permissions. Copying that code into twenty functions means twenty places to fix when the rule changes. A decorator writes it once and applies it with one line above any function.

You will also read decorators every day from Week 4 on: @app.get("/entries") in FastAPI registers a route, @pytest.fixture you already used on Day 10, @dataclass on Day 5, and tool definitions in agent frameworks are often decorators. Knowing how they work turns them from magic words into ordinary functions.

### How it works

Three facts build up to decorators. First, functions are objects: you can pass them as arguments and return them, exactly as in JavaScript. Second, an inner function can use variables from the function around it, even after the outer one has returned. That is a closure, the same mechanism React hooks rely on.

In [ ]:
# Step 1: functions are objects; a closure remembers its outer variables
def make_counter():
    count = 0  # lives on after make_counter returns, inside the closure

    def counter():
        nonlocal count  # "use the outer count", not a new local one
        count += 1
        return count

    return counter  # return the function itself, not counter()


tick = make_counter()
print(tick(), tick(), tick())  # -> 1 2 3
# JS: the same closure pattern as useState's setter capturing state

Third, a decorator is a function that takes a function and returns a new one, usually a wrapper that does something before and after calling the original. The line @timed above a def is only shorthand for fetch_entry = timed(fetch_entry). The wrapper takes *args, **kwargs so it can pass any arguments through, and @functools.wraps(func) copies the original's name and docstring onto the wrapper, so logs, tracebacks and help() still show the real name.

In [ ]:
# Step 2: a decorator takes a function and returns a wrapped version
import functools
import time


def timed(func):
    @functools.wraps(func)  # copy func's name and docstring to wrapper
    def wrapper(*args, **kwargs):  # accept ANY arguments and pass them on
        start = time.perf_counter()
        try:
            return func(*args, **kwargs)  # call the real function
        finally:  # runs even if func raises: we still log the time
            ms = (time.perf_counter() - start) * 1000
            print(f"{func.__name__} took {ms:.0f} ms")

    return wrapper


@timed  # same as: fetch_entry = timed(fetch_entry)
def fetch_entry(uid: str) -> dict:
    """Pretend to call the CMS."""
    time.sleep(0.05)
    return {"uid": uid}


print(fetch_entry("blt001"))
# -> fetch_entry took 50 ms
# -> {'uid': 'blt001'}
print(fetch_entry.__name__, "|", fetch_entry.__doc__)
# -> fetch_entry | Pretend to call the CMS.
# React: the same idea as a higher-order component, withAuth(Page)

When the decorator needs settings, such as how many attempts or which environment variables, you add one more layer: a function that takes the settings and returns the decorator. That is why @require_env("CMS_API_KEY") has brackets and @timed does not. Today's @retry(attempts=4, base_delay=0.5) uses exactly this three-layer shape.

In [ ]:
# Step 3: a decorator WITH arguments is a function that returns a decorator
import functools
import os


def require_env(*names: str):  # layer 1: takes the settings
    def decorator(func):  # layer 2: takes the function
        @functools.wraps(func)
        def wrapper(*args, **kwargs):  # layer 3: runs on every call
            missing = [n for n in names if not os.environ.get(n)]
            if missing:
                raise RuntimeError(f"missing env vars: {', '.join(missing)}")
            return func(*args, **kwargs)

        return wrapper

    return decorator


@require_env("CMS_API_KEY", "CMS_BASE_URL")  # note the () with arguments
def export_entries() -> str:
    return "exported"


os.environ["CMS_BASE_URL"] = "https://cms.test"
try:
    export_entries()
except RuntimeError as err:
    print(err)  # -> missing env vars: CMS_API_KEY
os.environ["CMS_API_KEY"] = "demo-key"
print(export_entries())  # -> exported

One more detail explains how FastAPI routes and webhook handler maps work: the decorator itself runs once, when Python executes the def line (usually at import). Only the wrapper runs on every call. A decorator can therefore register the function somewhere and return it unchanged.

In [ ]:
# A decorator runs ONCE, when the def line runs; the wrapper runs per call
def register(func):
    print("registering", func.__name__)  # happens at definition time
    HANDLERS[func.__name__] = func  # e.g. a webhook event -> handler map
    return func  # return the original: nothing is wrapped


HANDLERS = {}


@register  # -> registering entry_published  (printed right here)
def entry_published(payload: dict) -> str:
    return f"reindex {payload['uid']}"


print(HANDLERS["entry_published"]({"uid": "blt001"}))  # -> reindex blt001

### Real-world FDE example

A customer's CMS rate-limits writes to 10 per second and returns 429 with a Retry-After header when you go faster. Your migration has eleven functions that write to it. You put @retry(attempts=5, retry_on=(RateLimited, ServerBusy)) on each, and the retry policy lives in one place. When their platform team asks "how do you back off?", you open one 25-line file and the tests that prove it, instead of eleven copies that have drifted apart.

### Common mistakes

- Forgetting @functools.wraps(func). Every decorated function then reports its name as wrapper, which makes logs and error reports useless (Check 3).

- Forgetting to return the result of func(*args, **kwargs) in the wrapper. The decorated function then always returns None.

- Writing @retry without brackets for a decorator that takes arguments. Python passes your function in as attempts, and the error appears in a confusing place. Factories need @retry() even with all defaults.

- Retrying everything. A 401 or a validation error will fail the same way every time; retrying only delays the real error and can lock an account. Retry only errors that can pass on their own: 429, 503, timeouts.

- Putting slow work (a network call) in the decorator's outer layers. That code runs at import time, once, not on each call.

## Concept 4: Context managers and the with statement

### Why it matters for an FDE

Integration code opens things that must be closed: files, network sessions, database connections, locks, temporary folders, sandbox records created for a test. If an exception skips the cleanup, you leak connections until the customer's database refuses new ones, or you leave a half-written file that the next job reads as if it were complete. A context manager guarantees the cleanup runs, whatever happens inside the block.

### How it works

You have used with open(...) as f: since Day 4. The with statement calls the object's __enter__ method at the start (its return value goes to the name after as) and __exit__ at the end, on every way out: the block finishing, a return, or an exception. __exit__ receives the exception details, or three None values if there was no error. If it returns True, the exception is swallowed; return False (or nothing) to let it continue. In JavaScript you would write try { ... } finally { cleanup() }; in React, it is the cleanup function returned from useEffect.

In [ ]:
# A context manager class: __enter__ runs at "with", __exit__ at the end
import time


class Timer:
    def __enter__(self):
        self.start = time.perf_counter()
        return self  # this value is what "as t" receives

    def __exit__(self, exc_type, exc, tb):
        # called on EVERY exit: normal end, return, or an exception
        self.ms = (time.perf_counter() - self.start) * 1000
        print(f"block took {self.ms:.0f} ms, error: {exc_type}")
        return False  # False = do not hide the error, let it propagate


with Timer() as t:
    time.sleep(0.02)
# -> block took 20 ms, error: None

try:
    with Timer():
        raise ValueError("bad entry")
except ValueError as err:
    print("caught:", err)
# -> block took 0 ms, error: <class 'ValueError'>
# -> caught: bad entry

Writing a class is verbose for most cases. contextlib.contextmanager turns a generator function with exactly one yield into a context manager: the code before yield is the setup, the with block runs at the yield, and the code after it is the cleanup. Wrap the yield in try so that the cleanup also runs on errors. This is the same shape as a Day 10 fixture with yield, and that is not a coincidence: pytest fixtures are built on the same idea.

The example below is a reusable version of Day 4's safe save: write to a temporary file, and swap it in only if everything worked. Today's atomic_csv follows this pattern.

In [ ]:
# @contextmanager: write a context manager as a generator with ONE yield
import os
from contextlib import contextmanager
from pathlib import Path


@contextmanager
def atomic_write(path: Path):
    """Write to a temp file; replace the real file only on success."""
    tmp = path.with_suffix(path.suffix + ".tmp")
    f = open(tmp, "w", encoding="utf-8")
    try:
        yield f  # the body of the "with" block runs here
    except BaseException:
        f.close()
        tmp.unlink()  # failed: delete the half-written temp file
        raise  # and re-raise, so the caller still sees the error
    else:
        f.close()
        os.replace(tmp, path)  # success: swap it in (Day 4's safe save)


target = Path("contacts.json")
target.write_text("OLD", encoding="utf-8")

with atomic_write(target) as f:
    f.write("NEW")
print(target.read_text(encoding="utf-8"))  # -> NEW

try:
    with atomic_write(target) as f:
        f.write("HALF")
        raise RuntimeError("CMS went down mid-export")
except RuntimeError:
    pass
print(target.read_text(encoding="utf-8"))  # -> NEW  (old file kept)
print(sorted(p.name for p in Path(".").glob("contacts*")))
# -> ['contacts.json']  (no .tmp left behind)

Many objects you already use are context managers. Using them in a with block is the habit to build.

In [ ]:
# The with statement you already use: files, locks, sessions, tempdirs
import tempfile
from pathlib import Path

import requests  # Day 6

with tempfile.TemporaryDirectory() as tmp:  # folder deleted at the end
    p = Path(tmp) / "a.txt"
    p.write_text("hi", encoding="utf-8")
    print(p.exists())  # -> True
print(p.exists())  # -> False  (cleanup ran when the block ended)

with requests.Session() as session:  # closes its connections at the end
    print(type(session).__name__)  # -> Session

# Two managers in one with, closed in reverse order (last opened first)
with (
    open("in.txt", "w", encoding="utf-8") as a,  # () allowed: Python 3.10+
    open("out.txt", "w", encoding="utf-8") as b,
):
    print(a.closed, b.closed)  # -> False False
print(a.closed, b.closed)  # -> True True

### Real-world FDE example

A nightly job writes products.csv, and the customer's search indexer picks up that file at 02:00. One night the CMS times out halfway through the export. With a plain open("products.csv", "w") the indexer reads a file with half the products, and 3,000 items vanish from the shop's search until someone notices. With atomic_csv, the failed run leaves yesterday's complete file in place, the job exits with code 1, and the alert tells the on-call engineer what happened. Same code size, very different incident.

### Common mistakes

- Writing cleanup after the yield in a @contextmanager without try/finally (or try/except). If the block raises, the cleanup line never runs.

- Swallowing errors by accident: an except in the context manager that does not re-raise hides every failure of that type from the caller (Check 4).

- Returning a lazy object (a generator, a cursor) from inside a with block. The resource closes on return, before anyone reads from it (Check 5).

- Opening files without with "because the script is short". In a long-running service, every missed close is a leaked file handle.

- Putting two yield statements in a @contextmanager function. It must yield exactly once.

## Check your understanding (predict first, then run)

Type this into your scratch folder. Write a prediction for every numbered part before you run anything. If one surprises you, find the line responsible and explain it in the notebook.

In [ ]:
# Day 11 check: write a prediction for each numbered part, then run
from contextlib import contextmanager
from pathlib import Path

# 1 generators are one-shot
sizes = (len(w) for w in ["sso", "seo", "cdn"])
print(sum(sizes), sum(sizes))

# 2 laziness: what prints, and in what order?
def numbers():
    for n in [1, 2, 3]:
        print("make", n)
        yield n

first = next(n for n in numbers() if n > 1)
print("first:", first)

# 3 a decorator without functools.wraps
def shout(func):
    def wrapper(*args, **kwargs):
        return func(*args, **kwargs).upper()
    return wrapper

@shout
def greet(name):
    """Say hello."""
    return f"hello {name}"

print(greet("rohit"), greet.__name__, greet.__doc__)

# 4 does the error escape the with block?
@contextmanager
def quiet():
    try:
        yield
    except KeyError:
        print("swallowed")

with quiet():
    {}["missing"]
print("after 4")

# 5 a generator returned from inside a with block
Path("t.txt").write_text("a\nb\n", encoding="utf-8")

def read_lines(path):
    with open(path, encoding="utf-8") as f:
        return (line.strip() for line in f)

try:
    print(list(read_lines("t.txt")))
except ValueError as err:
    print("ValueError:", err)

- Check 1: what are the two sums, and why are they different?

- Check 2: which "make" lines are printed before "first: 2", and why does the generator stop where it does?

- Check 3: what are the name and the docstring of greet, and what one line fixes them?

- Check 4: does the KeyError escape the with block? What is printed?

- Check 5: what does read_lines give you, and how do you fix it so that it still streams?

## Hands-on task (about 2.5 hours): retry decorator and streaming export

The scenario: a customer moving off a legacy CMS gives you a JSON Lines export of every entry. Their platform lead says: "The full file is 5 GB and our job runner has 512 MB of memory. Our CMS API also throws 503s during deploys. Show me an exporter that streams, never leaves a half-written CSV, and a retry helper we can put on our API calls."

Before you start: git switch main && git pull, then git switch -c day11. The starters are in day11/: retry.py, stream_export.py, test_day11.py, dsa.py, plus export_sample.jsonl (12 lines with a blank line, two broken lines, drafts and a missing locale) and make_big_export.py (complete, a helper that writes a large fake export). Each function has a comment block with its rules and HINT lines, and a body of raise NotImplementedError("TODO").

Part A, retry.py (45 min). Write backoff_delays() and the decorator factory retry(), then the six retry tests in test_day11.py. Use sleep=waits.append in tests so nothing really waits.

Part B, stream_export.py (95 min). Build the generator stages read_jsonl, only_published, to_rows and batched_rows, the context manager atomic_csv, then run_export and main. Write the six export tests as you go, one stage at a time. Finish with the streaming proof on a 200,000-line file.

```bash
# Streaming proof (acceptance 4): run inside day11/ once the task is done
python3 make_big_export.py big_export.jsonl 200000
# -> wrote 200000 lines to big_export.jsonl (50 MB)
python3 -c "
import tracemalloc
from pathlib import Path
import stream_export as s
tracemalloc.start()  # starts counting every byte Python allocates
r = s.run_export(Path('big_export.jsonl'), Path('big.csv'), 500)
print(r['read'], len(r['bad_lines']), r['published'], r['batches'])
print(tracemalloc.get_traced_memory()[1] // 1024, 'KB peak')
"
# -> 199960 40 180000 360
# -> 577 KB peak      (your number will differ; it must stay under 5 MB)
```

Acceptance criteria:

- python3 retry.py prints exactly the lines shown in its __main__ block, with the retry lines on stderr.

- python3 stream_export.py export_sample.jsonl out.csv prints read: 9, bad lines: [5, 7], published: 6, batches: 2, wrote: out.csv and exits with 0; out.csv has the header and 6 rows, titles stripped, blt010 with locale en-us.

- Every stage in stream_export.py is a generator, and no line calls list() on file data. Calling read_jsonl(...) alone reads nothing.

- The streaming proof above shows a peak under 5 MB. big_export.jsonl and big.csv are in .gitignore, not committed.

- If the export fails halfway, no .tmp file is left and an existing out.csv is unchanged (a test proves it).

- A missing source file prints error: export file not found: ... to stderr and exits with 1; wrong arguments print usage and exit with 2.

- python3 -m pytest day11 -v from the repo root: 12 or more tests (parametrized cases count), all green, and --durations=3 shows nothing at 0.1 s or slower.

- Mutation check (Day 10): remove @functools.wraps, then make retry also retry PermissionError, then remove the blank-line check in read_jsonl. Each time at least one test must go red. Put the code back and note which test caught each bug.

- The work goes in on the branch day11 in at least two commits (retry; streaming export), pushed, with a pull request that has What, Why, How to test and Open items.

Hints (no solutions, try first):

- Start each stage with its test. A generator test can be as small as assert list(only_published([...])) == [...] with two hand-written dicts.

- If you are stuck on the decorator, first write retry without arguments (two layers, like timed), with attempts fixed at 3. Add the outer layer once that works.

- Write the three layers' return lines first (return wrapper, return decorator), then fill in the bodies. Most decorator bugs are a missing return.

- except retry_on as err: works with a tuple of classes, the same as except (A, B):.

- To count records while streaming, a four-line generator that increments a dict value and yields each item is enough. Do not count with len(); generators have no length.

- For atomic_csv, copy the structure of atomic_write from Concept 4 and change only what is opened and what is yielded.

- If the memory check shows tens of MB, search your code for list(, .readlines() or json.load(: one of them is loading the whole file.

## DSA practice (30 min)

Both problems are in day11/dsa.py with signatures, examples and hints. Both are natural generators, so they practise today's main idea.

- Moving average of a stream. moving_average([1, 2, 3, 4, 5], 3) yields 2.0, 3.0, 4.0, the average of each window of 3. The input may be a generator you can read only once. Hint: keep a running total and a collections.deque(maxlen=k); add the new number and subtract the one that falls out, so each step is constant work instead of re-summing the window.

- Flatten a nested list. flatten([1, [2, [3, 4]], [], 5]) yields 1, 2, 3, 4, 5. Hint: for each item, if it is a list, let a recursive call handle it with yield from; otherwise yield the item. Think about why the empty list needs no special case.

## Quiz (3 quick questions)

- A teammate's export does entries = list(read_jsonl(path)) and then loops over entries. It works on the 10 MB sample and is killed in production. Explain why, and what you change.

- Your @retry decorator is on a function that creates an order in the customer's system. The first call times out, but the order was actually created. What goes wrong when you retry, and what would you ask the customer's API team about?

- What is the difference between try/finally written inline and a context manager? Give one reason to prefer the context manager in a codebase you hand over to a customer.

## What you achieved today

You now know what a for loop does underneath, why some objects can only be looped over once, and how to write generators that process files of any size with flat memory. You can write decorators, including ones with arguments, and you know the three-layer shape that FastAPI and pytest use. You can write context managers that clean up on every path, including the atomic write pattern that keeps customer files safe. Your retry decorator and streaming exporter are reusable pieces you will put into Project 1 and Project 2.

Tomorrow (Day 12): project structure, pyproject.toml and uv, logging, ruff and mypy, used to turn cms_export into an installable package that passes lint and type checks.

## When you're done

Paste into the Forward Deployment Engineer project chat as "Day 11 review": your recap answers and check-your-understanding predictions next to the real outputs, retry.py, stream_export.py, test_day11.py, the output of python3 -m pytest day11 -v, the memory check output, the three mutation results, dsa.py, the link to the pull request, and your quiz answers. Say which concept felt least natural; tomorrow's recap starts there.

## Recap answers

*Your answers:*
